# IOAI — 2026 Local Mock Transport (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/transport.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-transport.zip', '/tmp/trans.zip')
    with zipfile.ZipFile('/tmp/trans.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/transport.csv' if os.path.exists('data/transport.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Transport — 대중교통 GPS 분석 (RoAI/ONIA 2026 Mock)

차량 GPS 텔레메트리(`transport.csv`)로 3 서브태스크:
1. (결정적) 고유 차량 수 / 차종 수
2. (군집) 위치로 도시 군집 (DBSCAN)
3. (군집) 차종 10의 야간(23~03시) 좌표로 차고지 3곳 (KMeans)

**제출**: `submission.csv` 롱포맷 `subtaskID, Value1, Value2`.
**채점(로컬)**: ST1 exact, ST2 레퍼런스 DBSCAN 대비 ARI, ST3 차고지 centroid 매칭거리. (GT 없는 비지도.)
**데이터**: `data/transport.csv`. 원본: platform.olimpiada-ai.ro/competitions/7

> **베이스라인(거친 첫 시도)**: ST1 카운트는 정확히, ST2 도시군집은 *k 를 잘못 추정(KMeans k=5)*, ST3 차고지는 *야간 필터 없이* KMeans → 점수가 낮습니다.
> 개선 포인트: ST2 는 도시 수를 모르므로 **DBSCAN**(밀도기반), ST3 는 **야간(23~03시)만 필터**해서 KMeans. 모범답안은 Solution/ 참고.
> ※ 데이터가 커서(86MB) 제출은 작업폴더 최상위에 `submission.csv` 로 저장합니다.

In [ ]:
import pandas as pd, numpy as np
from sklearn.cluster import KMeans

root_path = "data"
df = pd.read_csv(f"{root_path}/transport.csv")
df.head()

In [ ]:
# ST1 (결정적): 고유 차량 수 / 차종 수
subtask11 = df["id"].nunique()
subtask12 = df["vehicle_type"].nunique()
print(subtask11, subtask12)

In [ ]:
# ST2 (거친 군집): 차량별 중앙 좌표에 KMeans(k=5 추정) — TODO: 도시 수 모르니 DBSCAN 권장
vc = df.groupby("id")[["latitude","longitude"]].median().reset_index()
vc["city"] = KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(vc[["latitude","longitude"]])
subtask2 = vc[["id","city"]]

In [ ]:
# ST3 (거친 차고지): 차종 10 전체 좌표에 KMeans(k=3) — TODO: 야간(23~03시)만 필터해야 정확
type10 = df[df["vehicle_type"]==10][["latitude","longitude"]].copy()
type10["depot"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(type10)
depots = type10.groupby("depot")[["latitude","longitude"]].mean().reset_index(drop=True).sort_values("latitude").reset_index(drop=True)

In [ ]:
# 제출(롱포맷): subtaskID, Value1, Value2 — 작업폴더 최상위에 저장
def build_df(sid, a1, a2):
    if sid != 1:
        return pd.DataFrame({"subtaskID":[sid]*len(a1), "Value1": np.asarray(a1), "Value2": np.asarray(a2)})
    return pd.DataFrame({"subtaskID":[sid], "Value1":[a1], "Value2":[a2]})
submission_df = pd.concat([
    build_df(1, subtask11, subtask12),
    build_df(2, subtask2["id"].values, subtask2["city"].values),
    build_df(3, depots["latitude"].values, depots["longitude"].values),
])
submission_df.to_csv("submission.csv", index=False)
submission_df.head()

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)